In [ ]:
!pip install torch torchvision
!pip install albumentations

In [ ]:
!pip install dataset

In [ ]:
!pip install torch

In [ ]:
!pip install albumentations

In [ ]:
!pip install dataset

### loading your data, preprocessing it, and applying augmentations on-the-fly.

In [25]:
import os
import glob
import torch
import rasterio
import numpy as np
from torch.utils.data import Dataset
import albumentations as A
from torch.utils.data import DataLoader

class GlacialLakeDataset(Dataset):
    """
    Custom Dataset for glacial lake segmentation.
    Handles varying chunk sizes by padding them to a target size.
    """

    def __init__(self, image_dir, mask_dir, augmentations=None, target_size=256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.augmentations = augmentations
        self.target_size = target_size # <-- New: Define a target size

        # --- File pairing logic (your code is perfect, no changes needed) ---
        self.image_files = sorted(glob.glob(os.path.join(image_dir, "*.tif")))
        self.pairs = []
        for img_path in self.image_files:
            filename = os.path.basename(img_path)
            if "input_stack_chunk" in filename:
                parts = filename.split("_")
                if len(parts) >= 5:
                    X = parts[1]
                    Y = parts[-1].replace(".tif", "")
                    key = f"LISS3_{X}_lake_mask_chunk_{Y}.tif"
                    mask_path = os.path.join(mask_dir, key)
                    if os.path.exists(mask_path):
                        self.pairs.append((img_path, mask_path))

    def __len__(self):
        return len(self.pairs)

    # Inside the GlacialLakeDataset class

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
    
        # --- ✅ FIX STARTS HERE ---
        # We'll read the image and ALSO store its data type
        with rasterio.open(img_path) as src:
            image = src.read()
            img_dtype = src.profile['dtype'] # <-- Store the dtype here
    
        with rasterio.open(mask_path) as src:
            mask = src.read(1)
        # --- ✅ FIX ENDS HERE ---
        
        # --- Padding logic (no changes needed) ---
        num_channels, original_height, original_width = image.shape
        padded_image = np.zeros((num_channels, self.target_size, self.target_size), dtype=np.float32)
        padded_mask = np.zeros((self.target_size, self.target_size), dtype=np.float32)
        padded_image[:, :original_height, :original_width] = image
        padded_mask[:original_height, :original_width] = mask
    
        image = padded_image
        mask = padded_mask
    
        # --- ✅ USE THE STORED DTYPE HERE ---
        max_val = np.iinfo(img_dtype).max if np.issubdtype(img_dtype, np.integer) else 1.0
        image = image / max_val
        
        # Binarize the mask (no changes needed)
        mask = (mask > 0).astype(np.float32)
        
        if self.augmentations:
            augmented = self.augmentations(image=image.transpose(1, 2, 0), mask=mask)
            image = augmented['image'].transpose(2, 0, 1)
            mask = augmented['mask']
    
        if mask.ndim == 2:
            mask = np.expand_dims(mask, axis=0)
    
        return {
            "image": torch.from_numpy(image).float(),
            "mask": torch.from_numpy(mask).float()
        }
    

# --- Your main script to test it ---
if __name__ == '__main__':
    TRAIN_IMG_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\input_stack\train"
    TRAIN_MSK_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\lake_mask\train"
    
    BATCH_SIZE = 4
    NUM_WORKERS = 0

    train_transforms = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
    ])

    print("📦 Loading datasets with padding...")
    # Pass the target_size to the dataset
    train_dataset = GlacialLakeDataset(TRAIN_IMG_DIR, TRAIN_MSK_DIR, train_transforms, target_size=256)

    if len(train_dataset) > 0:
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS
        )

        print("🧪 Verifying one batch...")
        try:
            data_batch = next(iter(train_loader))
            images = data_batch['image']
            masks = data_batch['mask']

            print("✅ Verification successful!")
            print(f"🖼️  Images batch shape: {images.shape}")
            print(f"🛡️  Masks batch shape:  {masks.shape}")
        except RuntimeError as e:
            print(f"🔥 Verification failed after fixes: {e}")
    else:
        print("❌ No data found.")

📦 Loading datasets with padding...
🧪 Verifying one batch...
✅ Verification successful!
🖼️  Images batch shape: torch.Size([4, 8, 256, 256])
🛡️  Masks batch shape:  torch.Size([4, 1, 256, 256])


### Calling Data augmentation class while trianing , below is a sample

In [26]:
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2
#from dataset import GlacialLakeDataset  # Use only in .py; comment this out in Jupyter

# --- 1. Define Correct Paths ---
TRAIN_IMG_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\input_stack\train"
TRAIN_MSK_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\lake_mask\train"

VAL_IMG_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\input_stack\validate"
VAL_MSK_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\lake_mask\validate"

# --- 2. Define Augmentations ---
train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
])

val_transforms = None

# --- 3. Create Datasets ---
train_dataset = GlacialLakeDataset(
    image_dir=TRAIN_IMG_DIR,
    mask_dir=TRAIN_MSK_DIR,
    augmentations=train_transforms
)

val_dataset = GlacialLakeDataset(
    image_dir=VAL_IMG_DIR,
    mask_dir=VAL_MSK_DIR,
    augmentations=val_transforms
)

# --- 4. Print Stats ---
print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Validation samples: {len(val_dataset)}")

# --- 5. Verify One Sample ---
if len(train_dataset) > 0:
    sample = train_dataset[0]
    image_tensor = sample["image"]
    mask_tensor = sample["mask"]

    print("\n🔍 Sample verification:")
    print(f"   🔸 Image tensor shape: {image_tensor.shape}")
    print(f"   🔸 Image tensor dtype: {image_tensor.dtype}")
    print(f"   🔸 Mask tensor shape:  {mask_tensor.shape}")
    print(f"   🔸 Mask tensor dtype:  {mask_tensor.dtype}")
else:
    print("\n⚠️ Training dataset is empty! Check file names or directory paths.")


✅ Training samples: 1628
✅ Validation samples: 204

🔍 Sample verification:
   🔸 Image tensor shape: torch.Size([8, 256, 256])
   🔸 Image tensor dtype: torch.float32
   🔸 Mask tensor shape:  torch.Size([1, 256, 256])
   🔸 Mask tensor dtype:  torch.float32


### Data Loader

This script will import your custom dataset, create instances for your training, validation, and test sets, and then wrap them in DataLoader

In [28]:
import os
import torch
import albumentations as A
from torch.utils.data import DataLoader

# Comment out this import in Jupyter notebooks
# from dataset import GlacialLakeDataset  

# --- Updated Dataset Class ---
import rasterio
import numpy as np
from torch.utils.data import Dataset

class GlacialLakeDataset(Dataset):
    """
    Dataset class for glacial lake segmentation with LISS3 imagery.
    Automatically matches input_stack and lake_mask files using (X, Y) key.
    """

    def __init__(self, image_dir, mask_dir, augmentations=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.augmentations = augmentations

        self.pairs = []
        image_files = sorted(os.listdir(image_dir))

        for img_file in image_files:
            if "input_stack_chunk" not in img_file or not img_file.endswith(".tif"):
                continue

            parts = img_file.split("_")
            if len(parts) >= 5:
                X = parts[1]
                Y = parts[-1].replace(".tif", "")
                mask_filename = f"LISS3_{X}_lake_mask_chunk_{Y}.tif"
                img_path = os.path.join(image_dir, img_file)
                mask_path = os.path.join(mask_dir, mask_filename)

                if os.path.exists(mask_path):
                    self.pairs.append((img_path, mask_path))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        with rasterio.open(img_path) as src:
            dtype = src.profile['dtype']
            max_val = np.iinfo(dtype).max if np.issubdtype(dtype, np.integer) else 1.0
            image = src.read() / max_val

        with rasterio.open(mask_path) as src:
            mask = src.read(1)
            mask = (mask > 0).astype(np.float32)

        if self.augmentations:
            augmented = self.augmentations(image=image.transpose(1, 2, 0), mask=mask)
            image = augmented['image'].transpose(2, 1, 0)
            mask = augmented['mask']

        if mask.ndim == 2:
            mask = np.expand_dims(mask, axis=0)

        return {
            "image": torch.from_numpy(image).float(),
            "mask": torch.from_numpy(mask).float()
        }

# --- Main block ---
if __name__ == '__main__':
    # --- 1. Paths and Hyperparameters ---
    TRAIN_IMG_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\input_stack\train"
    TRAIN_MSK_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\lake_mask\train"

    VAL_IMG_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\input_stack\validate"
    VAL_MSK_DIR = r"C:\Users\Rochan\Desktop\Coding\Glacial_Lale_Detection_BAH-2025\data\LISS3\segments\lake_mask\validate"

    BATCH_SIZE = 4
    NUM_WORKERS = 0  # Use 0 for Jupyter

    # --- 2. Augmentations ---
    train_transforms = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
    ])
    val_transforms = None

    # --- 3. Instantiate Dataset ---
    print("📦 Loading datasets...")
    train_dataset = GlacialLakeDataset(TRAIN_IMG_DIR, TRAIN_MSK_DIR, train_transforms)
    val_dataset = GlacialLakeDataset(VAL_IMG_DIR, VAL_MSK_DIR, val_transforms)

    print(f"✅ Training samples: {len(train_dataset)}")
    print(f"✅ Validation samples: {len(val_dataset)}")

    # --- 4. Create DataLoaders ---
    if len(train_dataset) > 0:
        print("🔄 Creating DataLoaders...")
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        # --- 5. Verify One Batch ---
        print("🧪 Verifying training batch...")
        data_batch = next(iter(train_loader))
        images = data_batch['image']
        masks = data_batch['mask']

        print(f"🖼️  Images shape: {images.shape}")
        print(f"🛡️  Masks shape:  {masks.shape}")
        print(f"📏 Image dtype: {images.dtype}")
        print(f"📏 Mask dtype:  {masks.dtype}")
    else:
        print("❌ No data found. Please check your file paths and naming.")


📦 Loading datasets...
✅ Training samples: 1628
✅ Validation samples: 204
🔄 Creating DataLoaders...
🧪 Verifying training batch...
🖼️  Images shape: torch.Size([4, 8, 256, 256])
🛡️  Masks shape:  torch.Size([4, 1, 256, 256])
📏 Image dtype: torch.float32
📏 Mask dtype:  torch.float32


###  The U-Net Model

Here is a standard implementation of the U-Net model in PyTorch. It's composed of two main parts:

DoubleConv: A small helper module that applies two sequential convolution-batchnorm-ReLU layers, which is a repeating block in the U-Net.

UNET: The main class that builds the encoder (downsampling path), bottleneck, and decoder (upsampling path) using the DoubleConv blocks and skip connections.

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """(Convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        # x1 is from the upsampling path, x2 is the skip connection from the downsampling path
        x1 = self.up(x1)
        
        # Pad x1 if its size does not match x2
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        
        # Concatenate the skip connection tensor (x2) with the upsampled tensor (x1)
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    """Final 1x1 convolution to produce the output segmentation map."""
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class UNET(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(UNET, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        # Encoder Path
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)

        # Decoder Path
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        # --- Encoder ---
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        # --- Decoder ---
        # The output of each 'down' block is passed as a skip connection to the corresponding 'up' block
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

# --- Verification Script ---
if __name__ == '__main__':
    # Your image has 8 channels (Bands + Indices) and the mask is 1 channel
    in_channels = 8
    out_channels = 1 
    batch_size = 4
    
    # Create a dummy input tensor with the same shape as your data loader's output
    dummy_input = torch.randn(batch_size, in_channels, 256, 256)
    
    # Instantiate the model
    model = UNET(n_channels=in_channels, n_classes=out_channels)
    
    # Pass the dummy input through the model
    output = model(dummy_input)
    
    # Print the shapes to verify
    print("✅ Model Instantiated")
    print(f"Input shape:  {dummy_input.shape}")
    print(f"Output shape: {output.shape}")

    # The output shape should match the input shape in H and W dimensions
    assert dummy_input.shape[0] == output.shape[0]
    assert dummy_input.shape[2] == output.shape[2]
    assert dummy_input.shape[3] == output.shape[3]
    assert output.shape[1] == out_channels
    print("✅ Shape verification successful!")

✅ Model Instantiated
Input shape:  torch.Size([4, 8, 256, 256])
Output shape: torch.Size([4, 1, 256, 256])
✅ Shape verification successful!


### From U-Net to Attention U-Net

The Purpose: The Attention U-Net architecture is designed to focus on important regions while suppressing irrelevant background information. As your proposal mentions, these gates learn to highlight the most salient features being passed from the encoder to the decoder, which is crucial for improving segmentation accuracy in complex areas.



How it Works: The attention gate takes two inputs:

The feature map from the encoder (e.g., x1).

The gating signal from the deeper decoder layer (e.g., x from up1).

It uses the gating signal to produce a "mask" or an "attention map" that it applies to the encoder's feature map. This action effectively tells the decoder to "pay more attention" to specific parts of the skip connection.

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# The DoubleConv, Down, and OutConv classes remain exactly the same as before.
# I've included them here so you can copy the whole file easily.

class DoubleConv(nn.Module):
    """(Convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

# --- New Attention Gate Module ---
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionGate, self).__init__()
        # Convolution for the gating signal (from decoder)
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        # Convolution for the skip connection signal (from encoder)
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        # Final convolution to create the attention map (alpha)
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, g, x):
        # g: Gating signal from the decoder path
        # x: Skip connection from the encoder path
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        
        # Multiply the skip connection 'x' by the attention map 'psi'
        return x * psi

# --- Updated 'Up' Block with Attention ---
class Up(nn.Module):
    """Upscaling then double conv with attention"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True) if bilinear else nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        
        # Create the Attention Gate
        # F_g: Gating signal channels, F_l: Skip connection channels
        self.att = AttentionGate(F_g=in_channels // 2, F_l=in_channels // 2, F_int=in_channels // 4)
        
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        # x1 is from the upsampling path, x2 is the skip connection
        x1 = self.up(x1)
        
        # Get the attention-weighted skip connection
        x2 = self.att(g=x1, x=x2)
        
        # Pad x1 if its size does not match x2
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        
        # Concatenate and apply convolutions
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

# --- Final Attention UNET Class ---
class AttentionUNET(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(AttentionUNET, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        factor = 2 if bilinear else 1

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

# --- Verification Script ---
if __name__ == '__main__':
    in_channels = 8
    out_channels = 1 
    batch_size = 4
    
    dummy_input = torch.randn(batch_size, in_channels, 256, 256)
    
    # Instantiate the new AttentionUNET model
    model = AttentionUNET(n_channels=in_channels, n_classes=out_channels)
    
    output = model(dummy_input)
    
    print("✅ Attention U-Net Model Instantiated")
    print(f"Input shape:  {dummy_input.shape}")
    print(f"Output shape: {output.shape}")

    assert dummy_input.shape[0] == output.shape[0]
    assert output.shape[1] == out_channels
    assert dummy_input.shape[2] == output.shape[2]
    assert dummy_input.shape[3] == output.shape[3]
    print("✅ Shape verification successful!")

✅ Attention U-Net Model Instantiated
Input shape:  torch.Size([4, 8, 256, 256])
Output shape: torch.Size([4, 1, 256, 256])
✅ Shape verification successful!


### Building the Training Engine

Let's quickly build the components needed to train your AttentionUNET. We'll need:

A Loss Function: We'll start with PyTorch's BCEWithLogitsLoss. It's a standard and numerically stable choice for binary segmentation.

An Optimizer: We'll use Adam, which is a robust, all-around great choice.

A Training Loop: The core script that will handle epochs, batches, forward passes, loss calculation, and backpropagation.